# Modular Dynamics of Financial Market Networks — a replication

Silva, Comin, Peron, Rodrigues, Ye, Wilson, Hancock & Costa (2015), [arXiv:1501.05040v3](https://arxiv.org/abs/1501.05040).

**The paper's claim.** Build one stock-correlation network per day over 25 years. Detect *communities* — groups of stocks more densely connected to each other than to the rest. Then throw the network away, keeping only the community sizes and the **mixing matrix Π** (how many edges run between each pair of communities), and generate a fresh random network from that summary alone. The claim is that this compressed description reproduces the real network's topology, and that crises show up as the market abandoning its community structure.

**What this notebook is.** Every cell below *loads* a stored result and draws it. Nothing is computed here — the pipeline runs in `experiments/exp1..8`, and this is the reading of it.

**How to read it.** Sections §0–§5 reproduce the paper. §6 is where the replication earns its keep: the corrections the paper's own construction requires and does not apply. §9 is the verdict table.

In [ ]:
# Setup. Works from any interpreter and any working directory.
import importlib.util, os, subprocess, sys

def _find_root(start):
    # walk up looking for paper3_market_modularity/, so the notebook does not
    # care whether the kernel started in the repo root or in this folder
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isfile(os.path.join(p, "run_phases.py")):
            return p
        sub = os.path.join(p, "paper3_market_modularity")
        if os.path.isfile(os.path.join(sub, "run_phases.py")):
            return sub
        p = os.path.dirname(p)
    raise RuntimeError("could not find paper3_market_modularity/ from " + os.path.abspath(start))

PROJECT_ROOT = _find_root(os.getcwd())
os.chdir(PROJECT_ROOT)
for p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "experiments")):
    if p not in sys.path:
        sys.path.insert(0, p)

# this notebook only reads stored artifacts, so the list is short; pyarrow is
# what pandas needs to open the .parquet series and is easy to be missing
REQUIRED = {"numpy": "numpy>=1.24", "pandas": "pandas>=2.0", "pyarrow": "pyarrow>=14"}
missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"kernel is missing {len(missing)} package(s); installing into {sys.executable}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    except subprocess.CalledProcessError:
        # a system or homebrew python refuses pip installs (PEP 668)
        venv = os.path.join(os.path.dirname(PROJECT_ROOT), ".venv", "bin", "python")
        raise SystemExit(
            "pip refused to install into this interpreter.\n"
            f"Switch the notebook kernel to the project environment:\n  {venv}\n"
            f"or install by hand:\n  {sys.executable} -m pip install {' '.join(missing)}")

import json
import numpy as np, pandas as pd
from IPython.display import Image, display

R = os.path.join(PROJECT_ROOT, "results")
F = os.path.join(PROJECT_ROOT, "figures")
def phase(n, s): return json.load(open(os.path.join(R, f"phase{n}_{s}.json")))
def series(name): return pd.read_parquet(os.path.join(R, "series", name))
def fig(name): display(Image(filename=os.path.join(F, name)))
print(f"interpreter  {sys.executable}")
print(f"project      {PROJECT_ROOT}")
print("loaded artifacts from", R)

Artifacts loaded. Nothing above recomputed anything — the paths are all reads.

## §1 The data

The paper uses 348 NYSE stocks with complete history from January 1986 to February 2011. That universe cannot be rebuilt: the candidate list it drew from no longer exists. Ours comes from a *currently listed* NYSE directory, which carries a second survivorship filter — a smaller pool (1,575 vs 3,799) but more survivors.

*Log return*: `Y(t) = ln P(t) − ln P(t−1)`, the day's proportional price change. *Adjusted close*: price corrected for splits and dividends, so a 2-for-1 split does not read as a −50% return.

In [ ]:
u = phase(0, "universe"); q = phase(1, "returns")
print(f"candidates screened      : {u['n_candidates']}")
print(f"survived completeness    : {u['n_survivors']}")
print(f"survived quality gates   : {q['n_tickers']} stocks x {u['n_days_final']} days")
print(f"  ({u['panel_start']} to {u['panel_end']}; paper: 348 x 6008)")
print(f"pooled excess kurtosis   : {q['pooled_excess_kurtosis']:.1f}"
      f"   median per-ticker sd: {q['median_ticker_std']:.4f}")
fig("fig1_returns.png")

Returns are fat-tailed, as every financial series is. The bottom panel is the mean pairwise correlation — the crisis signal the paper's construction is about to discard.

## §2 From correlations to networks

For each day, correlate every pair of stocks over the trailing 30 returns, then keep the strongest 10% of pairs as edges. This is **fixed-density thresholding**: the cutoff τ is re-solved daily so the edge count never changes.

That has a consequence the paper does not draw out. Because the density is held constant, **τ itself becomes a discarded measurement** — and it is a market-stress signal.

In [ ]:
n = phase(2, "networks")
print(f"{n['n_windows']} networks x {n['n_edges_per_day']} edges, "
      f"mean degree {n['mean_degree']:.2f} = f(N-1) exactly")
print(f"tau: mean {n['tau_mean']:.3f}, peak {n['tau_max']:.3f} on {n['tau_argmax_date']}")
print(f"i.i.d. noise through the same pipeline: tau = {n['noise_reference']['tau_mean']:.3f}")
fig("fig2_networks.png")

τ peaks at 0.861 on 1987-10-19 — Black Monday, and the global maximum over 25 years.

**The first warning.** Pure i.i.d. Gaussian noise put through this exact pipeline gives τ = 0.241, against a real *typical* day of 0.444. With 30 observations per pair, a correlation matrix is mostly estimation error, and the quietest days in the sample are barely distinguishable from noise.

## §3 Communities

*Modularity* Q measures how well a partition explains a network: how many more edges fall inside groups than would by chance. *Louvain* is the algorithm that searches for the best partition. It is **stochastic** — it depends on the order it visits nodes — so we run it ten times per day.

In [ ]:
c = phase(3, "communities"); s = series("communities.parquet")
print(f"Q = {c['q_dyn']['mean']:.4f} +/- {c['q_dyn']['sd']:.4f}; "
      f"k median {c['k']['median']:.0f}, max {c['k']['max']}")
print(f"Black Monday: Q -> {c['black_monday']['q_min']:.4f}, "
      f"k -> {c['black_monday']['k_max']}, "
      f"{c['black_monday']['n_singleton_max']} singleton communities")
fig("fig3_blackmonday.png")

On Black Monday the network shatters: modularity hits its 25-year low and the community count explodes to 156, of which 152 are single isolated stocks. The paper's Fig. 3 reproduces.

### §3b How much of the modularity series is the algorithm?

The paper runs Louvain once per day and reads the resulting series as a record of the market. Our ten seeds let us ask how much of that series would change if only the random seed changed.

In [ ]:
sn = c["seed_noise"]; dg = c["diagnostics"]
print(f"seed-to-seed spread on one day : {sn['range_median']:.5f}")
print(f"day-to-day move                : {sn['day_move_median']:.5f}")
print(f"ratio                          : {sn['ratio_seed_range_to_day_move']:.2f}x, "
      f"seed noise wins on {sn['frac_days_seed_range_exceeds_day_move']:.0%} of days")
fig("fig5b_partition_decay.png")

The seed-to-seed spread on a *single day* is 2.09× the typical day-to-day move. Any feature of the modularity series finer than that is the algorithm, not the market.

The left panel adds a second finding: a partition's explanatory power has a half-life of about 14 trading days. The paper's lagged modularity uses a 100-day-old partition, which is far out on the flat part of that curve.

The right panel is the more serious one. **Louvain on pure-noise networks scores Q = 0.2246, against the real market's 0.2215.** The *level* of modularity is what structureless data produces here. Only its variation carries information.

## §4 The two null models

**Community model (the paper's Eq. 2):** keep the community sizes and Π, discard everything else, reconnect each pair of stocks with the probability their two communities imply.

**Configuration model:** keep only each stock's *degree* (its number of edges) and rewire at random. This is the baseline — anything it reproduces is explained by the degree distribution alone.

In [ ]:
nl = phase(4, "nulls")
cm, cf = nl["community_model"], nl["configuration_model"]
print(f"Eq.2 reproduces its own planted modularity: MAE {cm['q_planted_vs_real_mae']:.5f}")
print(f"noise floor (configuration model): Q = {cf['q_mean']:.4f} "
      f"(paper draws ~{cf['paper_noise_floor']:.2f})")
fig("fig6a_noise_floor.png")

The configuration model gives Q = 0.1065, reproducing the floor the paper draws in Fig. 6(a1) and never states. The real market sits well above it — the orange community-null line tracks the blue real line so closely it has to be drawn underneath to stay visible.

### §4b The control the comparison is missing

The market's modularity exceeds its degree-sequence null by a wide margin, and that gap is the paper's evidence of modular organisation. The control is to ask what gap *structureless data* produces under the same pipeline.

In [ ]:
x = nl["noise_excess"]
print(f"real market : Q {nl['real']['q_mean']:.4f} over floor {cf['q_mean']:.4f}"
      f" -> excess {x['real_excess']:+.4f}")
print(f"i.i.d. noise: Q {x['q_noise']:.4f} over floor {x['q_noise_null']:.4f}"
      f" -> excess {x['noise_excess']:+.4f}")
print(f"-> noise reproduces {x['frac_of_real_excess_explained']:.0%} of the real excess")

**80% of the market's excess modularity is reproduced by data containing no communities at all.**

The mechanism: a correlation matrix is positive semi-definite, so if A correlates with B and with C, then B and C are forced to correlate too. Thresholding one therefore produces a graph that is *transitive by construction*, and transitive graphs are modular. A degree-sequence null cannot control for this, because rewiring destroys the geometry along with the structure. This bounds every result that follows.

## §5 The eight measures

The paper characterises each network with eight quantities and compares the real series against both nulls. Two of them — average shortest path and betweenness — are undefined when the network splits into pieces, and the paper never says how it handles that. We compute both readings.

In [ ]:
m = phase(5, "measures")
for k in m["measures"]:
    v = m["per_measure"][k]
    print(f"  {k:<16} real {v['real_mean']:>10.4f}   community {v['comm_mean']:>10.4f}"
          f"   configuration {v['conf_mean']:>10.4f}")
fig("fig7_measures.png")

The community null (orange) tracks the real series (blue) closely on most measures, while the configuration null (aqua) does not. This is the paper's central figure, and at this level of inspection it reproduces.

## §6 The correction that matters

Every number in the paper's comparison tables is a correlation between two series of ~6,000 daily points. But each point is a 30-day window slid by one day, so **consecutive points share 29 of their 30 days of data.**

Those are not 6,000 independent observations. They are closer to 6,000/30 ≈ 200. Two heavily overlapping, heavily smoothed series will correlate highly almost regardless of what generated them. Recomputing on windows that share *no* data is the honest version of the same statistic.

In [ ]:
t = pd.read_csv(os.path.join(R, "tables", "fig6_scoring.csv"))
sc = phase(6, "scoring"); ov = sc["overlap_correction"]
cols = ["measure","model","era","rho","rho_nonoverlapping","paper_rho"]
display(t[t.era!="all"][cols].round(3).to_string(index=False))
print(f"\nmean rho over 32 cells: {ov['mean_rho_overlapping']:.3f} ->"
      f" {ov['mean_rho_nonoverlapping']:.3f}  ({ov['mean_drop']:+.3f})")
fig("fig8_overlap.png")

Mean correlation across all 32 cells falls from 0.783 to 0.777 when the overlap is removed — a drop on 12 of 32 cells, worst case 0.164.

In the dumbbell figure, the filled dot is the paper's number and the hollow dot is the corrected one. The length of each connector is how much of the paper's evidence was window overlap.

### §6b Three more things the tables need

**Normalized MSE:** the paper prints squared differences ranging from 0.0003 to 2,522 in one table. Betweenness is O(100) and transitivity is O(0.1) — the raw numbers mostly measure the units. Dividing by the real series' variance makes them comparable.

**Independence:** eight measurements are presented as eight facts. Are they?

**Compression:** the community summary stores `k + k(k+1)/2` numbers against the degree sequence's `N`. Is it actually smaller?

In [ ]:
e5, d1 = sc["inter_measure"], sc["compression"]
print(f"mean |rho| between the 8 measures: {e5['mean_abs_offdiag']:.3f}; "
      f"{e5['n_components_for_90pct']}/8 components carry 90% of variance")
print(f"compression cheaper on {d1['frac_days_cheaper']:.1%} of days, but only "
      f"{d1['frac_days_cheaper_sharp_crisis']:.1%} of crisis days "
      f"(break-even k={d1['break_even_k']:.0f}, observed max {d1['k_max']})")
fig("fig8b_inter_measure.png"); fig("fig8c_compression.png")

The eight measures are far from independent, so the eight agreements are not eight pieces of evidence.

The compression claim **inverts exactly where the paper's story lives**: on ordinary days the community summary is genuinely smaller, but during crises the network shatters into hundreds of singleton communities, Π explodes, and describing the market by its communities costs more than listing every stock's degree.

## §7 PCA

The paper's closing figure projects all three families into two dimensions and reads the picture as agreement. It reports no explained-variance ratios and no loadings, and asserts without measurement that the first component is essentially time.

In [ ]:
pc = phase(7, "pca")
print(f"explained variance: PC1 {pc['explained_variance_ratio'][0]:.1%}, "
      f"PC1+PC2 {pc['cumulative_variance'][1]:.1%}")
print(f"corr(PC1, date) = {pc['pc1_vs_time_corr']:+.3f}")
cl = pc["classifier"]
print(f"classifier telling the 3 families apart: {cl['three_class_accuracy']:.1%} "
      f"(chance {cl['chance_three_class']:.1%})")
fig("fig9_pca.png")

corr(PC1, date) = +0.439 — the "PCA1 is time" claim, finally given a number.

And the separation the paper describes as agreement is measurable: a simple classifier tells the three families apart 99% of the time against 33% chance. A null model genuinely indistinguishable from the real networks would sit at chance.

## §8 The detector the paper promises

Figure 1 of the paper is a methodology flowchart. Step (e) is a labelled box reading *"crisis detection evaluation"*. The paper contains no detection rule, no threshold, no hit rate and no false-alarm rate.

Here is one, scored against the 17 crisis windows **pre-registered in Phase 0 — written down before any result existed**, so they cannot have been tuned to flatter the answer.

In [ ]:
x8 = phase(8, "extensions")
for k, v in sorted(x8["detectors"].items(), key=lambda kv: -kv[1]["sharp"]["auc"]):
    print(f"  {k:<18} AUC {v['sharp']['auc']:.3f}   F1 {v['sharp']['f1']:.3f}")
print(f"base rate: {x8['base_rate_sharp']:.1%} of days are in a sharp window")
fig("fig10b_detector.png"); fig("fig10c_event_study.png")

The event-study panel settles the paper's internal contradiction (§C1): the abstract says crises destroy community structure, while the Fig. 4 text says modularity *increases* around crises. Aligning all pre-registered onsets shows which reading the data supports.

Notably, τ(t) — the quantity fixed-density thresholding throws away — is competitive with modularity as a crisis detector, at a fraction of the computational cost.

## §9 Verdict

Claim by claim, with the gap attributed. The full table, including every number, is in `results/REPLICATION_REPORT.md`.

In [ ]:
print(open(os.path.join(R, "REPLICATION_REPORT.md")).read())

The paper's mechanical claims reproduce. Its interpretive claims are where the corrections bite: the overlap correction removes much of the statistical confidence, the noise control removes most of the modularity excess, and the compression claim inverts during exactly the crises the paper is about.